In [1]:
### Load data
import pandas as pd   
# Prepare the historical panel (df1)
df1 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119.xlsx") 
df1 = df1.rename(columns={'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'})   # Rename the historical cooperation-mode column
df1.loc[df1['Both'] == 1, ['Conflict', 'Cooperation']] = 1
#df1['Cooperation'] = df1[label].any(axis=1).astype(int)   # Some panel rows have an active mode but Cooperation is not 1.
# Prepare future inputs (df2) and merge the mode indicators 
df2 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\E_ssp_predication\ssp1-5_future_data.xlsx", sheet_name='ssp5')  
df2 = df2[df2['year'] == 2050]  
# Prepare future cooperation-mode indicators (df3)
df3 = pd.read_csv(r"E:\AExperiment\11Water\E_RF_Data\D_strategy_onehot\20241218_updated_future_data_with_modes.csv")
df3.rename(columns={
    'ccode_iso3': 'country_code',
    'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'
}, inplace=True)   # Rename the country identifier and cooperation-mode column
df3 = df3[df3['year'] == 2050]   # Retain observations for 2050 only
# Define the selected columns
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative Planning and Adaptation Strategies'] 
# Select country_code and the mode-indicator columns from df3  
df3_labels = df3[['country_code'] + label]  
# Left-join df3_labels to df2  
df2 = pd.merge(df2, df3_labels, on='country_code', how='left')  
#df2.to_excel('D:\\OneDrive\\Desktop\\df2.xlsx', index=False)   # Optional intermediate output for inspection

In [2]:
### Combine historical and future inputs into df
import pandas as pd 
df2.rename(columns={  
    # Keep 'year' and 'export' unchanged when the column names already match 
    'country': 'Country',
    'country_code': 'ccode_iso3',
    'population_density_ssp5': 'population_density',  # For an SSP1 run, use the corresponding SSP1 field
    'GDP per capita_ssp5(constant 2015 US$)': 'gdp_percapita',  # For an SSP1 run, use the corresponding SSP1 field  
    'rural_population_ssp5(%)': 'rural_population',  # For an SSP1 run, use the corresponding SSP1 field
    'precipitation_ssp5': 'precipitation',   # For an SSP1 run, use the corresponding SSP1 field 
    'bws_ssp5': 'bws_score'}, inplace=True)   # For an SSP1 run, use the corresponding SSP1 field   
# Concatenate historical and future rows 
df = pd.concat([df1, df2], ignore_index=True, sort=True)
### Identify high-conflict countries for 2050 
df4 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119\Clustered_Conflict_Probability_241119.xlsx")
# Select rows assigned to Cluster 1  
high_conflict_country = df4[df4['Cluster'] == 1]  
# Extract the ISO3 country identifiers  
high_conflict_country = high_conflict_country['ccode_iso3'].tolist()  
### Set Conflict to 1 for high-conflict countries in 2050
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Conflict'] = 1
#df.to_excel('D:\\OneDrive\\Desktop\\df.xlsx', index=False)   # Optional intermediate output for inspection

In [3]:
'''
### Carry the 2019 Average_conflict value forward to 2050
# Select observations for 2019  
df_2019_Average_conflict = df[df['year'] == 2019][['ccode_iso3', 'Average_conflict']]  
# Create a dictionary of the 2019 country-level values  
conflict_dict = df_2019_Average_conflict.set_index('ccode_iso3')['Average_conflict'].to_dict()  
# Update the 2050 observations  
df.loc[df['year'] == 2050, 'Average_conflict'] = df[df['year'] == 2050]['ccode_iso3'].map(conflict_dict)  
'''
### Project Average_conflict to 2050 using a time-series model
import pandas as pd  
from statsmodels.tsa.holtwinters import ExponentialSmoothing   
# Initialize a DataFrame for the 2050 forecasts  
predictions_2050 = pd.DataFrame(columns=['ccode_iso3', 'Average_conflict_2050'])  
# Forecast each country in high_conflict_country  
for country in high_conflict_country:  
    country_data = df[(df['ccode_iso3'] == country) & (df['year'].between(1951, 2019))]  
    if not country_data.empty:  
        print(f"Data for {country}:")  
        print(country_data[['Average_conflict']])  # Display the historical Average_conflict series for this country  
        # Fit exponential smoothing to the country-level time series  
        model = ExponentialSmoothing(country_data['Average_conflict'], trend='add', seasonal=None).fit()  
        # Forecast the series from 2020 through 2050  
        forecast = model.forecast(31)  # Forecast 31 steps beyond 2019 to reach 2050  
        print(f"Forecast for {country} from 2020 to 2050:")  
        print(forecast)  # Display the forecast  
        # Check that the forecast is available and whether its final value is negative  
        if not forecast.empty and forecast.iloc[-1] < 0:  
            forecast.iloc[-1] = 0  # Truncate a negative 2050 forecast to zero  
        # Check that a forecast is available after truncation  
        if not forecast.empty:  
            # Retain the forecast for 2050  
            forecast_2050 = forecast.iloc[-1]  
            new_prediction = pd.DataFrame({'ccode_iso3': [country], 'Average_conflict_2050': [forecast_2050]})  
            predictions_2050 = pd.concat([predictions_2050, new_prediction], ignore_index=True)  
        else:  
            print(f"No forecast available for {country}")  
# Merge the forecasts back into df  
df = df.merge(predictions_2050, on='ccode_iso3', how='left')  
# Update the projected values in the 2050 rows  
# Use the existing 2050 rows and update Average_conflict  
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Average_conflict'] = df['Average_conflict_2050']  
# Remove the temporary forecast column  
df.drop(columns=['Average_conflict_2050'], inplace=True)  
# Optionally inspect the updated DataFrame  
# print(df)  
# Save the results to an Excel file  
#df.to_excel('D:\\OneDrive\\Desktop\\time_series_2050.xlsx', index=False)

Data for AFG:
    Average_conflict
0           0.000000
1           0.000000
2           0.000000
3           0.000000
4           0.000000
..               ...
64          0.061538
65          0.060606
66          0.059701
67          0.073529
68          0.072464

[69 rows x 1 columns]
Forecast for AFG from 2020 to 2050:
69    0.075619
70    0.078412
71    0.081205
72    0.083998
73    0.086791
74    0.089584
75    0.092377
76    0.095170
77    0.097963
78    0.100756
79    0.103549
80    0.106342
81    0.109135
82    0.111928
83    0.114721
84    0.117514
85    0.120308
86    0.123101
87    0.125894
88    0.128687
89    0.131480
90    0.134273
91    0.137066
92    0.139859
93    0.142652
94    0.145445
95    0.148238
96    0.151031
97    0.153824
98    0.156617
99    0.159410
dtype: float64
Data for DZA:
     Average_conflict
142          0.000000
143          0.000000
144          0.000000
145          0.000000
146          0.000000
..                ...
206          0.153846
207  

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8860\763572420.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions_2050 = pd.concat([predictions_2050, new_prediction], ignore_index=True)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next versi

Data for USA:
       Average_conflict
10585          0.000000
10922          0.000000
10923          0.000000
10924          0.000000
10925          0.250000
...                 ...
10985          0.250000
10986          0.276923
10987          0.272727
10988          0.283582
10989          0.294118

[69 rows x 1 columns]
Forecast for USA from 2020 to 2050:
69    0.299036
70    0.303955
71    0.308873
72    0.313792
73    0.318710
74    0.323629
75    0.328548
76    0.333466
77    0.338385
78    0.343303
79    0.348222
80    0.353140
81    0.358059
82    0.362977
83    0.367896
84    0.372814
85    0.377733
86    0.382652
87    0.387570
88    0.392489
89    0.397407
90    0.402326
91    0.407244
92    0.412163
93    0.417081
94    0.422000
95    0.426919
96    0.431837
97    0.436756
98    0.441674
99    0.446593
dtype: float64
Data for YEM:
       Average_conflict
11243          0.000000
11244          0.000000
11245          0.000000
11246          0.000000
11247          0.000000
.

D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction resul

In [4]:
'''
###Replace the 2050 precipitation and bws_score inputs with their 2019 values
# Select the 2019 reference observations  
df_2019_prebws = df[df['year'] == 2019]  
# Iterate over the 2050 rows and replace the selected inputs  
for index, row in df[df['year'] == 2050].iterrows():  
    # Find the corresponding country observation for 2019  
    country_2019_data = df_2019_prebws[df_2019_prebws['ccode_iso3'] == row['ccode_iso3']]  
    if not country_2019_data.empty:  
        # Replace the selected input values  
        df.at[index, 'precipitation'] = country_2019_data['precipitation'].values[0]  
        df.at[index, 'bws_score'] = country_2019_data['bws_score'].values[0]  
# Inspect the DataFrame after replacement  
#df.to_excel('D:\\OneDrive\\Desktop\\output.xlsx', index=False)
#print(df)
'''

"\n###Replace the 2050 precipitation and bws_score inputs with their 2019 values\n# Select the 2019 reference observations  \ndf_2019_prebws = df[df['year'] == 2019]  \n# Iterate over the 2050 rows and replace the selected inputs  \nfor index, row in df[df['year'] == 2050].iterrows():  \n    # Find the corresponding country observation for 2019  \n    country_2019_data = df_2019_prebws[df_2019_prebws['ccode_iso3'] == row['ccode_iso3']]  \n    if not country_2019_data.empty:  \n        # Replace the selected input values  \n        df.at[index, 'precipitation'] = country_2019_data['precipitation'].values[0]  \n        df.at[index, 'bws_score'] = country_2019_data['bws_score'].values[0]  \n# Inspect the DataFrame after replacement  \n#df.to_excel('D:\\OneDrive\\Desktop\\output.xlsx', index=False)\n#print(df)\n"

In [5]:
### Preprocess the data
import numpy as np  
from sklearn.preprocessing import StandardScaler
# Create potential_Conflict as a copy of the original Conflict indicator  
#df['potential_Conflict'] = df['Conflict']   # The 2050 test sample retains conflict-coded observations, so potential_Conflict would equal 1.
# Define the selected columns
SEdata = ['population_density', 'export', 'gdp_percapita', 'rural_population', 'precipitation', 'bws_score']
# Select the analysis fields  
df = df[['year', 'ccode_iso3', 'Country_count', 'Average_conflict', 'Conflict'] + SEdata + label]
# Clean and standardize the data  
#pd.set_option('future.no_silent_downcasting', True)  
df = df.replace('..', np.nan)  
df = df.dropna(subset=df.columns.difference(['Country_count']))
# Standardize the selected continuous predictors
scaler = StandardScaler()  
df[SEdata] = scaler.fit_transform(df[SEdata])  
### Remove observations with missing inputs
#df = df.dropna(subset=df.columns.difference(['Country_count']))   # Drop rows missing any selected field other than Country_count.
# Display the number of retained observations
num_rows = df.shape[0]  
print("Number of rows:", num_rows)  
#df.to_excel('D:\\OneDrive\\Desktop\\output.xlsx', index=False)   # Optional intermediate output for inspection

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8860\3253034828.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('..', np.nan)


Number of rows: 5982


In [6]:
### Define the outcome and predictor variables
# Outcome and predictor variables  
y = 'Country_count'
x = df[['Average_conflict'] + ['Conflict'] + SEdata + label]
### Define the training and future prediction samples
# Select the training sample
mask_train = df['year'] <= 2019  
x_train = x[mask_train]  
y_train = df[y][mask_train]
# Select future observations after 2019  
# Select the future prediction sample  
mask_test = df['year'] > 2019
x_test = x[mask_test]  
y_test = df[y][mask_test]

In [7]:
### Fit the Random Forest and compare scenario predictions
# Initialize the Random Forest regressor  
import pandas as pd  
from sklearn.ensemble import RandomForestRegressor  
rfr = RandomForestRegressor(random_state=42)  
rfr.fit(x_train, y_train)  
# Scenario 1 - Noncooperation: keep Conflict unchanged and set all mode indicators to 0  
x_test_case1 = x_test.copy()  
x_test_case1[label] = 0  
y_pred_case1 = rfr.predict(x_test_case1)  
# Store the predictions in new columns for the future sample  
df.loc[mask_test, 'Noncooperation'] = y_pred_case1  
# Scenario 2 - Cooperation: set Conflict to 0 and retain the mode indicators  
x_test_case2 = x_test.copy()  
x_test_case2['Conflict'] = 0  
y_pred_case2 = rfr.predict(x_test_case2)  
# Store the predictions in new columns for the future sample  
df.loc[mask_test, 'Cooperation'] = y_pred_case2  
# Scenario 3 - Mitigation: calculate Cooperation minus Noncooperation  
df.loc[mask_test, 'Mitigation'] = df.loc[mask_test, 'Cooperation'] - df.loc[mask_test, 'Noncooperation']  
### Calculate the means of Noncooperation, Cooperation, and Mitigation
mean_noncooperation = df.loc[mask_test, 'Noncooperation'].mean()
mean_cooperation = df.loc[mask_test, 'Cooperation'].mean()
mean_mitigation = df.loc[mask_test, 'Mitigation'].mean()
# Display the scenario results  
print(df[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Mitigation']])
print(f"Mean Non-Cooperation: {mean_noncooperation:.2f}\n"
      f"Mean Cooperation: {mean_cooperation:.2f}\n"
      f"Mean Mitigation: {mean_mitigation:.2f}")
#df[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Mitigation']].to_excel('D:\\OneDrive\\Desktop\\ssp5_avecof_prebws.xlsx', index=False)

       year ccode_iso3  Noncooperation  Cooperation  Mitigation
11557  2050        AFG            2.10         2.49        0.39
11559  2050        DZA            0.89         0.84       -0.05
11593  2050        CHN            0.89         1.04        0.15
11613  2050        ETH            0.72         0.97        0.25
11633  2050        HUN            0.51         0.05       -0.46
11635  2050        IND            1.75         1.77        0.02
11637  2050        IRN            1.40         1.47        0.07
11644  2050        JOR            1.87         2.09        0.22
11653  2050        LSO            1.17         1.09       -0.08
11667  2050        MEX            1.89         1.73       -0.16
11682  2050        NGA            1.68         1.41       -0.27
11688  2050        PSE            0.54         0.29       -0.25
11696  2050        PRI            0.36         0.04       -0.32
11715  2050        SOM            1.34         1.53        0.19
11718  2050        SSD            1.09  